# 01 · Universo

**Objetivo:** Definir os carros candidatos: FIPE de R$ 50–85 mil no mês mais recente, ano-modelo ≥ 2017 (sem zero km), só carros de passeio.

**Entradas:** histórico FIPE (fipeX, versão merged); `data/ref/utilitarios.csv`; `data/ref/modelo_base_excecoes.csv`; `data/ref/fabricantes.csv`

**Saídas:** `data/processed/universo.parquet` (um par código × ano-modelo por linha) e `data/processed/fipe_hist_universo.parquet` (histórico de preços só desses pares)

## Passos

- [x] Baixar o histórico e carregar só carros, só as colunas necessárias
- [x] Conferir a fonte: mês mais recente, zero km, chave única
- [x] Extrair atributos do nome (regex) e medir a cobertura
- [x] Definir `modelo_base` (primeiro token + exceções revisadas)
- [x] Padronizar o nome da fabricante (`fabricante`)
- [x] Aplicar `filtrar_universo` e revisar os excluídos
- [x] Checar duplicidade de (código FIPE, ano-modelo) e contagens
- [x] Salvar universo e histórico dos pares

## Setup

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # permite importar src/ a partir de notebooks/

import pandas as pd
from src.config import carregar_premissas, RAIZ
from src.dados import baixar_fipe, carregar_fipe_carros
from src.features import atributos_com_cache, cobertura, definir_modelo_base
from src.universo import adicionar_fabricante, filtrar_universo

pd.set_option("display.max_rows", 200)
p = carregar_premissas()

## 1. Carga

Usamos o arquivo **merged** do fipeX (nome atual do modelo em todo o histórico). O split `train` do
Hugging Face soma a versão original e a merged, então cada preço aparece duas vezes: por isso não
usamos `load_dataset`. O filtro `tipo_veiculo == carro` é aplicado já na leitura.

In [2]:
baixar_fipe(p)                 # só baixa se o arquivo ainda não estiver em data/raw
df = carregar_fipe_carros(p)
print(f"{len(df):,} linhas de carro | memória: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.info()

5,923,372 linhas de carro | memória: 0.17 GB
<class 'pandas.DataFrame'>
RangeIndex: 5923372 entries, 0 to 5923371
Data columns (total 8 columns):
 #   Column             Dtype   
---  ------             -----   
 0   codigo_fipe        category
 1   nome_marca         category
 2   nome_modelo        category
 3   sigla_combustivel  category
 4   ano_modelo         Int64   
 5   zero_km            bool    
 6   valor              float64 
 7   mesano_referencia  int32   
dtypes: Int64(1), bool(1), category(4), float64(1), int32(1)
memory usage: 158.6 MB


## 2. Conferências da fonte

- O mês mais recente precisa estar completo: comparamos o número de linhas com os meses anteriores.
- Zero km: no fipeX vem como `zero_km = True` com `ano_modelo` nulo (não como ano 32000).
- Chave: na FIPE o "ano-modelo" é ano **+ combustível**. No histórico antigo existe o mesmo
  código e ano com dois combustíveis; por isso a chave completa inclui `sigla_combustivel`.

In [3]:
print("Linhas nos últimos meses de referência:")
print(df["mesano_referencia"].value_counts().sort_index().tail(4).to_string())

print("\nzero_km × ano_modelo nulo:")
print(pd.crosstab(df["zero_km"], df["ano_modelo"].isna(), colnames=["ano_modelo nulo"]))

chave = ["codigo_fipe", "ano_modelo", "mesano_referencia"]
print("\nRepetições da chave (código, ano, mês):     ", df.duplicated(chave).sum())
print("Repetições incluindo o combustível na chave:", df.duplicated(chave + ["sigla_combustivel"]).sum())

Linhas nos últimos meses de referência:
mesano_referencia
202608    30433
202609    30473
202610    30550
202611    30604

zero_km × ano_modelo nulo:


ano_modelo nulo    False   True 
zero_km                         
False            5611180       0
True                   0  312192



Repetições da chave (código, ano, mês):      125111


Repetições incluindo o combustível na chave: 16


## 3. Atributos do nome (regex)

A extração roda só sobre os **nomes únicos** do mês mais recente (alguns milhares) e é juntada de
volta por `nome_modelo`. O cache em parquet é refeito sozinho se algum padrão mudar.

Atributo não encontrado fica **nulo**: por exemplo, a FIPE muitas vezes omite "Mec." em carros
manuais, então `transmissao` nula não quer dizer "não tem câmbio". Imputação, se necessária, é na etapa 02.

In [4]:
ultimo = df[df["mesano_referencia"] == df["mesano_referencia"].max()].copy()
ultimo["nome_modelo"] = ultimo["nome_modelo"].astype(str)

atributos = atributos_com_cache(ultimo["nome_modelo"], RAIZ / p["dados"]["cache_regex"])
ultimo = ultimo.merge(atributos, on="nome_modelo", how="left")

u = p["universo"]
na_faixa = (~ultimo["zero_km"] & (ultimo["ano_modelo"] >= u["ano_modelo_min"])
            & ultimo["valor"].between(u["fipe_min"], u["fipe_max"])).fillna(False)
nomes_faixa = ultimo.loc[na_faixa, "nome_modelo"].unique()
print(f"{len(atributos):,} nomes no mês mais recente; {len(nomes_faixa):,} na faixa de preço/ano")
cobertura(atributos).join(cobertura(atributos[atributos["nome_modelo"].isin(nomes_faixa)]),
                          lsuffix="_mes", rsuffix="_faixa")

Cache válido com 7,403 nomes; 0 novos para processar.


7,403 nomes no mês mais recente; 838 na faixa de preço/ano


C:\Users\ramon\AppData\Local\Temp\ipykernel_15540\150458331.py:12: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  cobertura(atributos).join(cobertura(atributos[atributos["nome_modelo"].isin(nomes_faixa)]),


,pct_preenchido_mes,pct_preenchido_faixa
motorizacao,88.8,97.9
combustivel_token,47.6,88.7
valvulas,45.4,85.2
transmissao,42.8,76.8
portas,28.9,47.7
tracao,17.2,5.6
carroceria_explicita,16.9,10.3
turbo,16.7,15.3
potencia_cv,15.5,3.2
injecao,13.0,4.3


In [5]:
# Amostra para conferência visual da extração
cols = ["nome_modelo", "motorizacao", "valvulas", "transmissao", "turbo", "combustivel_token",
        "carroceria_explicita", "cabine", "nome_modelo_limpo"]
atributos[atributos["nome_modelo"].isin(nomes_faixa)][cols].sample(15, random_state=0)

,nome_modelo,motorizacao,valvulas,transmissao,turbo,combustivel_token,carroceria_explicita,cabine,nome_modelo_limpo
3274,HB20 Ocean 1.6 Flex 16V 5p Aut.,1.6,16V,AUT,NaN,FLEX,NaN,NaN,HB20 OCEAN
1648,Punto BLACKMOTION Dual. 1.8 Flex 16v 5p,1.8,16V,NaN,NaN,FLEX,NaN,NaN,PUNTO BLACKMOTION DUAL
6036,LOGAN Life Flex 1.0 12V 4p Mec.,1.0,12V,MEC,NaN,FLEX,NaN,NaN,LOGAN LIFE
2180,Ka 1.5 Sedan SE 12V Flex 4p Mec.,1.5,12V,MEC,NaN,FLEX,SEDAN,NaN,KA SE
6165,STEPWAY Zen Flex 1.6 16V Mec.,1.6,16V,MEC,NaN,FLEX,NaN,NaN,STEPWAY ZEN
6261,XLV 1.6 16V 128cv Aut.,1.6,16V,AUT,NaN,NaN,NaN,NaN,XLV
2713,ONIX HATCH LT 1.0 8V FlexPower 5p Mec.,1.0,8V,MEC,NaN,FLEXPOWER,HATCH,NaN,ONIX LT
2873,SPIN PREMIER 1.8 8V Econo.Flex 5p Mec.,1.8,8V,MEC,NaN,ECONO.FLEX,NaN,NaN,SPIN PREMIER
5939,DUSTER Dynamique 1.6 Flex 16V Aut.,1.6,16V,AUT,NaN,FLEX,NaN,NaN,DUSTER DYNAMIQUE
6714,Fox Highline1.6 Flex 16V 5p,1.6,16V,NaN,NaN,FLEX,NaN,NaN,FOX HIGHLINE


## 4. `modelo_base`

Regra: primeiro token do nome. Exceções em `data/ref/modelo_base_excecoes.csv` para modelos de duas
palavras ou nomes enganosos. Exemplo: "DUSTER OROCH" é a picape Oroch, não o Duster. A lista de
exceções saiu da revisão dos primeiros tokens na faixa de preço.

In [6]:
excecoes = pd.read_csv(RAIZ / "data" / "ref" / "modelo_base_excecoes.csv")
ultimo["modelo_base"] = definir_modelo_base(ultimo["nome_modelo"], excecoes)
excecoes

,prefixo,modelo_base,motivo
0,DUSTER OROCH,OROCH,picape derivada do Duster; nome começa com DUSTER
1,GRAND C4,GRAND C4,modelo distinto do C4
2,GRAND SIENA,GRAND SIENA,modelo distinto do Siena
3,C4 CACTUS,C4 CACTUS,modelo distinto do C4
4,C4 LOUNGE,C4 LOUNGE,sedã distinto do C4 hatch
5,C4 PICASSO,C4 PICASSO,minivan distinta do C4
6,CLASSE B,CLASSE B,primeiro token é a palavra CLASSE
7,XC 60,XC60,FIPE separa XC e 60
8,SX4 S-CROSS,S-CROSS,mesmo modelo que S-CROSS
9,PALIO WEEK,PALIO WEEKEND,perua distinta do Palio (WEEK. e WEEKEND)


## 4b. Fabricante

A FIPE escreve a mesma fabricante de formas diferentes (ex.: o TIGGO aparece em "Caoa Chery" e em
"Caoa Chery/Chery"). A tabela `data/ref/fabricantes.csv` liga cada `nome_marca` a um nome padronizado.
Se aparecer uma marca nova sem correspondência, a função para com erro em vez de deixar nulo.

In [7]:
fabricantes = pd.read_csv(RAIZ / "data" / "ref" / "fabricantes.csv")
ultimo = adicionar_fabricante(ultimo, fabricantes)

# Só as marcas que mudam de nome, entre as presentes no mês mais recente
(ultimo[["nome_marca", "fabricante"]].astype(str).drop_duplicates()
 .query("nome_marca != fabricante").sort_values("fabricante").reset_index(drop=True))

,nome_marca,fabricante
0,ARROW,Arrow
1,ASTON MARTIN,Aston Martin
2,CHANA,Changan
3,CHANGAN,Changan
4,Caoa Changan,Changan
5,Caoa Chery,Chery
6,Caoa Chery/Chery,Chery
7,GM - Chevrolet,Chevrolet
8,EFFA,Effa
9,FEVER,Fever


## 5. Filtro do universo

`filtrar_universo` aplica a faixa (mês mais recente, usado, ano ≥ mínimo, FIPE na faixa) e marca o
motivo de exclusão de cada linha. Quando mais de um motivo vale, fica o primeiro desta ordem:
PCD → modelo utilitário → cabine → carroceria utilitária → token de carga no nome.

In [8]:
utilitarios = pd.read_csv(RAIZ / "data" / "ref" / "utilitarios.csv")["modelo_base"].tolist()
universo, excluidos = filtrar_universo(ultimo, p, utilitarios)
print(f"Universo: {len(universo):,} pares | excluídos: {len(excluidos):,}")
excluidos["motivo_exclusao"].value_counts()

Universo: 1,750 pares | excluídos: 269


motivo_exclusao
modelo utilitário (utilitarios.csv)    267
token de carga no nome                   2
Name: count, dtype: int64

In [9]:
# Revisão: quais modelos foram excluídos e por quê
excluidos.groupby(["motivo_exclusao", "modelo_base"]).size().rename("pares").to_frame()

pares
motivo_exclusao                     modelo_base       
modelo utilitário (utilitarios.csv) AMAROK           2
                                    BERLINGO         1
                                    BUGGY           46
                                    DELIVERY         1
                                    E.CO            21
                                    FIORINO         14
                                    JIMNY           12
                                    KANGOO           1
                                    MINITRUCK        3
                                    MONTANA          8
                                    OROCH           29
                                    PARTNER          7
                                    SAVEIRO         43
                                    STRADA          44
                                    TORO            16
                                    V21              4
                                    V22              6
                                    V25              5
                                    WAY              4
token de carga no nome              DOBLO            1
                                    E-JS1            1

In [10]:
# Revisão: excluídos por token/cabine/carroceria (casos mais sujeitos a falso positivo)
excluidos.loc[excluidos["motivo_exclusao"] != "modelo utilitário (utilitarios.csv)",
              ["nome_marca", "nome_modelo", "motivo_exclusao"]]

,nome_marca,nome_modelo,motivo_exclusao
5243,Fiat,Doblo Cargo 1.8 mpi Fire Flex 8V/16V 4p,token de carga no nome
14470,JAC,E-JS1 City Cargo Aut. (Elétrico),token de carga no nome


## 6. Conferências do universo

In [11]:
print("Pares (código, ano-modelo) repetidos:", universo.duplicated(["codigo_fipe", "ano_modelo"]).sum())
print("Modelos-base distintos:", universo["modelo_base"].nunique())
universo["valor"].describe().round(0)

Pares (código, ano-modelo) repetidos: 0
Modelos-base distintos: 115


count     1750.0
mean     67498.0
std       9694.0
min      50019.0
25%      59508.0
50%      67461.0
75%      75807.0
max      84915.0
Name: valor, dtype: float64

In [12]:
universo.groupby("fabricante").agg(
    pares=("codigo_fipe", "size"), modelos=("modelo_base", "nunique")
).sort_values("pares", ascending=False)

,pares,modelos
fabricante,,
Chevrolet,223,9
Hyundai,216,7
Renault,188,8
Volkswagen,162,13
Citroën,135,9
Fiat,127,11
Toyota,119,4
Ford,117,6
Peugeot,113,4


In [13]:
universo["ano_modelo"].value_counts().sort_index()

ano_modelo
2017    285
2018    284
2019    284
2020    280
2021    211
2022    152
2023    107
2024     72
2025     38
2026     29
2027      8
Name: count, dtype: Int64

In [14]:
universo["modelo_base"].value_counts()

modelo_base
HB20             87
ONIX             77
HB20S            69
ETIOS            60
DUSTER           53
YARIS            52
ARGO             49
2008             47
208              47
SPIN             46
C3               44
SANDERO          42
POLO             42
ECOSPORT         40
RENEGADE         38
ONIX PLUS        37
C4 CACTUS        36
VERSA            36
CRONOS           35
KA               32
FOX              31
CAPTUR           29
TIGGO            27
KICKS            27
FIESTA           26
HB20X            25
FIT              24
UP!              24
AIRCROSS         22
VITARA           22
C4 LOUNGE        21
VIRTUS           21
COBALT           20
CITY             20
CRETA            20
KWID             20
LOGAN            20
FOCUS            16
STEPWAY          16
MARCH            15
MOBI             14
PRISMA           14
ARRIZO           13
TRACKER          13
SENTRA           12
LANCER           11
308              11
SPACEFOX         11
VOYAGE           11
CRUZE   

## 7. Salvar

- `universo.parquet`: um par (código, ano-modelo) por linha, com preço do mês mais recente e atributos.
- `fipe_hist_universo.parquet`: todo o histórico de preço só desses pares, para as etapas 02 e 03
  não precisarem recarregar os ~6 milhões de linhas.

In [15]:
dir_proc = RAIZ / p["dados"]["dir_processed"]
dir_proc.mkdir(parents=True, exist_ok=True)
universo.to_parquet(dir_proc / "universo.parquet", index=False)

# Histórico ligado por (código, ano-modelo, combustível): a chave completa da FIPE.
chave_hist = ["codigo_fipe", "ano_modelo", "sigla_combustivel"]
pares = universo[chave_hist].astype({"codigo_fipe": str, "sigla_combustivel": str})
hist = df.astype({"codigo_fipe": str, "sigla_combustivel": str}).merge(pares, on=chave_hist)
hist.to_parquet(dir_proc / "fipe_hist_universo.parquet", index=False)

print(f"universo.parquet: {len(universo):,} linhas")
print(f"fipe_hist_universo.parquet: {len(hist):,} linhas, "
      f"de {hist['mesano_referencia'].min()} a {hist['mesano_referencia'].max()}")

universo.parquet: 1,750 linhas
fipe_hist_universo.parquet: 152,428 linhas, de 201603 a 202611


## Decisões e observações

- **Fonte:** arquivo `fipex-prices-latest-merged.parquet`. O split `train` do Hugging Face soma as
  versões original e merged (9.631.829 linhas cada) e duplicava todos os preços no notebook antigo.
- **Zero km:** no fipeX vem como `zero_km = True` e `ano_modelo` nulo; excluímos por essa coluna
  (e também por ano 32000, por segurança).
- **Chave:** na FIPE o ano-modelo é ano + combustível. No universo, (código, ano-modelo) já é único;
  o histórico é ligado ao universo por (código, ano-modelo, combustível).
- **Mês de referência:** o mais recente do arquivo, com contagem de linhas em linha com os meses anteriores.
- **Regex:** padrões em `src/features.py`; fronteiras que aceitam tokens com ponto ("TIP.", "SED.");
  atributo ausente = nulo. Cobertura na faixa: motorização ~98%, combustível ~89%, válvulas ~85%,
  transmissão ~77% (manuais muitas vezes sem "Mec.").
- **modelo_base:** primeiro token + exceções revisadas (`modelo_base_excecoes.csv`). O Onix sedã
  ("ONIX SEDAN/SED./SD.") vira `ONIX PLUS`, separado do hatch (decisão do autor em 2026-11-06).
- **fabricante:** nome padronizado da marca (`fabricantes.csv`, todas as marcas do mês mais recente).
  Junta, por exemplo, "Caoa Chery" e "Caoa Chery/Chery" em Chery, e "GM - Chevrolet" vira Chevrolet.
  `nome_marca` original é mantida.
- **Exclusões em `utilitarios.csv` revisadas pelo autor (2026-11-06):** AMAROK, BERLINGO, PARTNER,
  V21, V22, V25, MINITRUCK, DELIVERY, E.CO, BUGGY, e também WAY e JIMNY (fora de estrada).
  Elétricos (E-JS1, iEV, ZOE, iCar, Kwid E-Tech) ficam no universo.
- **PCD:** regra mantida; não há versões PCD na faixa no mês atual.
- **Doblo:** só a Doblo Cargo é excluída (token CARGO); as versões de passageiros ficam.